In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

In [ ]:
# ── Change team name here to analyse any IPL franchise ───────────────────────
TEAM = "Chennai Super Kings"

# For teams that changed official name add the alternate name(s) here.
# Example for RCB: TEAM_ALIASES = ["Royal Challengers Bengaluru"]
TEAM_ALIASES = []
# ─────────────────────────────────────────────────────────────────────────────

_names  = [TEAM] + TEAM_ALIASES
IN_LIST = ", ".join(f"'{t}'" for t in _names)
print(f"Analysing: {TEAM}  ({len(_names)} name variant(s) in lookup)")

## Match Record

In [ ]:
# IPL season-by-season win/loss record
q(f"""
SELECT
    mode(CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_1 ELSE m.team_2 END)           AS team,
    m.season,
    count(*)                                                                           AS played,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)                   AS won,
    sum(CASE WHEN m.has_result
              AND m.outcome_winner NOT IN ({IN_LIST}) THEN 1 ELSE 0 END)               AS lost,
    sum(m.is_tie::integer)                                                             AS tied,
    sum(m.is_no_result::integer)                                                       AS nr,
    round(
        sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0
        / nullif(sum(m.has_result::integer), 0), 1)                                    AS win_pct
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
GROUP BY m.season
ORDER BY m.season
""")

In [ ]:
# Overall IPL record by match stage (League vs Playoff)
q(f"""
SELECT
    coalesce(m.event_stage, 'League')                                                  AS stage,
    count(*)                                                                           AS played,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)                   AS won,
    sum(CASE WHEN m.has_result
              AND m.outcome_winner NOT IN ({IN_LIST}) THEN 1 ELSE 0 END)               AS lost,
    sum(m.is_tie::integer)                                                             AS tied,
    sum(m.is_no_result::integer)                                                       AS nr,
    round(
        sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0
        / nullif(sum(m.has_result::integer), 0), 1)                                    AS win_pct
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
GROUP BY coalesce(m.event_stage, 'League')
ORDER BY played DESC
""")

## Head-to-Head

In [ ]:
# All-time IPL head-to-head vs every opponent
q(f"""
SELECT
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END                 AS opponent,
    count(*)                                                                           AS played,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)                   AS won,
    sum(CASE WHEN m.has_result
              AND m.outcome_winner NOT IN ({IN_LIST}) THEN 1 ELSE 0 END)               AS lost,
    sum(m.is_tie::integer)                                                             AS tied,
    sum(m.is_no_result::integer)                                                       AS nr,
    round(
        sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0
        / nullif(sum(m.has_result::integer), 0), 1)                                    AS win_pct
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
GROUP BY opponent
ORDER BY played DESC, won DESC
""")

## Toss Analysis

In [ ]:
# Toss win rate, decision split, and toss-to-match-win conversion
q(f"""
SELECT
    count(*)                                                                                    AS matches,
    sum(CASE WHEN m.toss_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)                               AS toss_won,
    round(sum(CASE WHEN m.toss_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0 / count(*), 1)  AS toss_win_pct,
    sum(CASE WHEN m.toss_winner IN ({IN_LIST}) AND m.toss_decision = 'bat'   THEN 1 ELSE 0 END) AS chose_bat,
    sum(CASE WHEN m.toss_winner IN ({IN_LIST}) AND m.toss_decision = 'field' THEN 1 ELSE 0 END) AS chose_field,
    sum(CASE WHEN m.toss_winner     IN ({IN_LIST}) AND m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) AS won_toss_won_match,
    sum(CASE WHEN m.toss_winner NOT IN ({IN_LIST}) AND m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) AS lost_toss_won_match
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.has_result
""")

In [ ]:
# Win rate by toss decision (when team won the toss)
q(f"""
SELECT
    m.toss_decision                                                                              AS decision,
    count(*)                                                                                     AS times_chosen,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)                             AS won,
    round(sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0 / count(*), 1) AS win_pct
FROM main_marts.fct_match_results m
WHERE m.toss_winner IN ({IN_LIST})
  AND (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.has_result
GROUP BY m.toss_decision
""")

## Team Batting

In [ ]:
# Team batting performance by season (super overs excluded)
q(f"""
WITH innings AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(d.runs_total)                                                      AS team_total,
        sum(d.runs_extras)                                                     AS extras,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('retired hurt', 'retired out')
             THEN 1 ELSE 0 END)                                                AS wickets_lost,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary
             THEN 1 ELSE 0 END)                                                AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary
             THEN 1 ELSE 0 END)                                                AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batting_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
)
SELECT
    season,
    count(*)                         AS innings,
    sum(team_total)                  AS total_runs,
    round(avg(team_total), 1)        AS avg_score,
    max(team_total)                  AS highest,
    min(team_total)                  AS lowest,
    round(avg(wickets_lost), 1)      AS avg_wkts_lost,
    sum(fours)                       AS fours,
    sum(sixes)                       AS sixes
FROM innings
GROUP BY season
ORDER BY season
""")

In [ ]:
# 10 highest team innings scores
q(f"""
WITH innings AS (
    SELECT d.match_id, d.innings_number, d.match_date, m.season,
        CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END      AS vs,
        m.event_stage,
        sum(d.runs_total)                                                      AS team_total,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('retired hurt', 'retired out')
             THEN 1 ELSE 0 END)                                                AS wickets_lost
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batting_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, m.season,
             m.team_1, m.team_2, m.event_stage
)
SELECT season, match_date, vs, event_stage,
    (team_total::varchar || '/' || wickets_lost::varchar) AS score,
    team_total
FROM innings
ORDER BY team_total DESC LIMIT 10
""")

In [ ]:
# 10 lowest team innings scores (completed innings)
q(f"""
WITH innings AS (
    SELECT d.match_id, d.innings_number, d.match_date, m.season,
        CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END      AS vs,
        m.event_stage,
        sum(d.runs_total)                                                      AS team_total,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('retired hurt', 'retired out')
             THEN 1 ELSE 0 END)                                                AS wickets_lost
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batting_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, m.season,
             m.team_1, m.team_2, m.event_stage
)
SELECT season, match_date, vs, event_stage,
    (team_total::varchar || '/' || wickets_lost::varchar) AS score,
    team_total
FROM innings
WHERE wickets_lost >= 8
ORDER BY team_total ASC LIMIT 10
""")

## Team Bowling

In [ ]:
# Team bowling performance by season (super overs excluded)
q(f"""
WITH innings AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(d.runs_total)                                                      AS runs_conceded,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
             THEN 1 ELSE 0 END)                                                AS legal_del,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out',
                                            'obstructing the field')
             THEN 1 ELSE 0 END)                                                AS wkts_taken
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowling_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
)
SELECT
    season,
    count(*)                                                        AS innings,
    round(avg(runs_conceded), 1)                                    AS avg_conceded,
    min(runs_conceded)                                              AS best_defence,
    max(runs_conceded)                                              AS most_conceded,
    round(avg(wkts_taken), 1)                                       AS avg_wkts,
    round(sum(runs_conceded) * 6.0 / nullif(sum(legal_del), 0), 2)  AS economy
FROM innings
GROUP BY season
ORDER BY season
""")

In [ ]:
# 10 lowest totals conceded (best defensive bowling innings)
q(f"""
WITH innings AS (
    SELECT d.match_id, d.innings_number, d.match_date, m.season,
        CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END      AS vs,
        m.event_stage,
        sum(d.runs_total)                                                      AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out',
                                            'obstructing the field')
             THEN 1 ELSE 0 END)                                                AS wkts_taken
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowling_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, m.season,
             m.team_1, m.team_2, m.event_stage
)
SELECT season, match_date, vs, event_stage,
    (runs_conceded::varchar || '/' || wkts_taken::varchar) AS score_conceded,
    runs_conceded
FROM innings
ORDER BY runs_conceded ASC LIMIT 10
""")

## Top Performers

In [ ]:
# Top 20 run-scorers while playing for this team (IPL, super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS batter,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    sum(b.not_out::integer)                                                     AS no,
    sum(b.runs)                                                                 AS runs,
    max(b.runs)                                                                 AS hs,
    round(sum(b.runs)::double / nullif(sum(b.dismissals), 0), 2)                AS avg,
    sum(b.balls_faced)                                                          AS bf,
    round(sum(b.runs) * 100.0 / nullif(sum(b.balls_faced), 0), 2)               AS sr,
    sum(CASE WHEN b.runs >= 50 AND b.runs < 100 THEN 1 ELSE 0 END)              AS "50s",
    sum(CASE WHEN b.runs >= 100 THEN 1 ELSE 0 END)                              AS "100s",
    sum(b.fours)                                                                AS "4s",
    sum(b.sixes)                                                                AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY runs DESC
LIMIT 20
""")

In [ ]:
# Top 20 wicket-takers while bowling for this team (IPL, super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS bowler,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    (floor(sum(b.legal_deliveries) / 6)::integer
     || '.' || (sum(b.legal_deliveries) % 6))::varchar                         AS overs,
    sum(b.legal_deliveries)                                                     AS balls,
    sum(b.runs_conceded)                                                        AS runs,
    sum(b.wickets_credited)                                                     AS wkts,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wickets_credited), 0), 2) AS avg,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_deliveries), 0), 2)  AS econ,
    round(sum(b.legal_deliveries)::double
          / nullif(sum(b.wickets_credited), 0), 2)                              AS sr,
    sum(CASE WHEN b.wickets_credited >= 4 THEN 1 ELSE 0 END)                   AS "4w+",
    sum(CASE WHEN b.wickets_credited >= 5 THEN 1 ELSE 0 END)                   AS "5w+"
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY wkts DESC
LIMIT 20
""")

## Biggest Wins & Losses

In [ ]:
# 10 biggest wins by runs margin
q(f"""
SELECT
    m.season, m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END          AS vs,
    m.event_stage,
    m.outcome_by_runs                                                           AS won_by_runs
FROM main_marts.fct_match_results m
WHERE m.outcome_winner IN ({IN_LIST})
  AND (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.outcome_by_runs IS NOT NULL
ORDER BY m.outcome_by_runs DESC LIMIT 10
""")

In [ ]:
# 10 biggest wins by wickets margin
q(f"""
SELECT
    m.season, m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END          AS vs,
    m.event_stage,
    m.outcome_by_wickets                                                        AS won_by_wickets
FROM main_marts.fct_match_results m
WHERE m.outcome_winner IN ({IN_LIST})
  AND (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.outcome_by_wickets IS NOT NULL
ORDER BY m.outcome_by_wickets DESC LIMIT 10
""")

In [ ]:
# 10 biggest losses by runs margin
q(f"""
SELECT
    m.season, m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END          AS vs,
    m.event_stage,
    m.outcome_by_runs                                                           AS lost_by_runs
FROM main_marts.fct_match_results m
WHERE m.outcome_winner NOT IN ({IN_LIST})
  AND m.has_result
  AND (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.outcome_by_runs IS NOT NULL
ORDER BY m.outcome_by_runs DESC LIMIT 10
""")

In [ ]:
# 10 biggest losses by wickets margin
q(f"""
SELECT
    m.season, m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END          AS vs,
    m.event_stage,
    m.outcome_by_wickets                                                        AS lost_by_wickets
FROM main_marts.fct_match_results m
WHERE m.outcome_winner NOT IN ({IN_LIST})
  AND m.has_result
  AND (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.outcome_by_wickets IS NOT NULL
ORDER BY m.outcome_by_wickets DESC LIMIT 10
""")